# Importing the Dependencies

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# Data Collection & Analysis

In [2]:
# Load the dataset. Upload parkinsons_disease_data.csv to Colab before running this cell.
from pathlib import Path

data_path = Path('/content/parkinsons_disease_data.csv')
if not data_path.exists():
    from google.colab import files
    uploaded = files.upload()
    data_path = Path(next(iter(uploaded)))

parkinsons_data = pd.read_csv(data_path)
print("Dataset loaded successfully:", data_path.name)


Dataset loaded successfully: parkinsons_disease_data.csv


In [3]:
# Display the first five rows
parkinsons_data.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,FunctionalAssessment,Tremor,Rigidity,Bradykinesia,PosturalInstability,SpeechProblems,SleepDisorders,Constipation,Diagnosis,DoctorInCharge
0,3058,85,0,3,1,19.619878,0,5.108241,1.380660,3.893969,...,1.572427,1,0,0,0,0,0,0,0,DrXXXConfid
1,3059,75,0,0,2,16.247339,1,6.027648,8.409804,8.513428,...,4.787551,0,1,0,1,0,1,0,1,DrXXXConfid
2,3060,70,1,0,0,15.368239,0,2.242135,0.213275,6.498805,...,2.130686,1,0,0,0,1,0,1,1,DrXXXConfid
3,3061,52,0,0,0,15.454557,0,5.997788,1.375045,6.715033,...,3.391288,1,1,1,0,0,0,1,1,DrXXXConfid
4,3062,87,0,0,1,18.616042,0,9.775243,1.188607,4.657572,...,3.200969,0,0,0,1,0,1,0,0,DrXXXConfid


In [4]:
# Number of rows and columns
parkinsons_data.shape

(2105, 35)

In [5]:
# Dataset information
parkinsons_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2105 entries, 0 to 2104
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   PatientID                 2105 non-null   int64  
 1   Age                       2105 non-null   int64  
 2   Gender                    2105 non-null   int64  
 3   Ethnicity                 2105 non-null   int64  
 4   EducationLevel            2105 non-null   int64  
 5   BMI                       2105 non-null   float64
 6   Smoking                   2105 non-null   int64  
 7   AlcoholConsumption        2105 non-null   float64
 8   PhysicalActivity          2105 non-null   float64
 9   DietQuality               2105 non-null   float64
 10  SleepQuality              2105 non-null   float64
 11  FamilyHistoryParkinsons   2105 non-null   int64  
 12  TraumaticBrainInjury      2105 non-null   int64  
 13  Hypertension              2105 non-null   int64  
 14  Diabetes

In [6]:
# Check missing values in each column
parkinsons_data.isnull().sum()

,0
PatientID,0
Age,0
Gender,0
Ethnicity,0
EducationLevel,0
BMI,0
Smoking,0
AlcoholConsumption,0
PhysicalActivity,0
DietQuality,0


In [7]:
# Statistical summary of numeric columns
parkinsons_data.describe()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MoCA,FunctionalAssessment,Tremor,Rigidity,Bradykinesia,PosturalInstability,SpeechProblems,SleepDisorders,Constipation,Diagnosis
count,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,...,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000,2105.000000
mean,4110.000000,69.601900,0.492637,0.692637,1.337292,27.209493,0.296437,10.040413,5.016674,4.912901,...,15.094314,4.989694,0.431829,0.252732,0.207601,0.138717,0.295012,0.245131,0.296912,0.619477
std,607.805479,11.594511,0.500065,1.003827,0.895840,7.208099,0.456795,5.687014,2.890919,2.872115,...,8.643014,2.933877,0.495449,0.434682,0.405686,0.345733,0.456156,0.430267,0.457006,0.485631
min,3058.000000,50.000000,0.000000,0.000000,0.000000,15.008333,0.000000,0.002228,0.004157,0.000011,...,0.021191,0.001505,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3584.000000,60.000000,0.000000,0.000000,1.000000,20.782176,0.000000,5.150278,2.455703,2.478503,...,7.517160,2.415890,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,4110.000000,70.000000,0.000000,0.000000,1.000000,27.184571,0.000000,10.070337,5.031550,4.825187,...,14.963574,4.983227,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,4636.000000,80.000000,1.000000,1.000000,2.000000,33.462452,1.000000,14.829565,7.512795,7.381487,...,22.608362,7.484220,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,1.000000
max,5162.000000,89.000000,1.000000,3.000000,3.000000,39.999887,1.000000,19.988866,9.995255,9.995864,...,29.970107,9.992697,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [8]:
# Distribution of the target variable: Diagnosis
parkinsons_data['Diagnosis'].value_counts()

,count
Diagnosis,
1,1304
0,801


**Target column: `Diagnosis`**

- `1` = Parkinson's disease diagnosed
- `0` = Parkinson's disease not diagnosed

This dataset uses `Diagnosis`; it does not contain the `status` column from the older Parkinson's voice dataset.

In [9]:
# Mean values grouped by Diagnosis (numeric columns only)
parkinsons_data.groupby('Diagnosis').mean(numeric_only=True)


,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,UPDRS,MoCA,FunctionalAssessment,Tremor,Rigidity,Bradykinesia,PosturalInstability,SpeechProblems,SleepDisorders,Constipation
Diagnosis,,,,,,,,,,,,,,,,,,,,,
0,4143.732834,68.635456,0.481898,0.699126,1.332085,26.932601,0.293383,9.774184,4.968954,4.997137,...,72.683695,17.002809,5.831890,0.258427,0.149813,0.112360,0.073658,0.302122,0.250936,0.282147
1,4089.279141,70.195552,0.499233,0.688650,1.340491,27.379577,0.298313,10.203948,5.045987,4.861158,...,119.064114,13.921994,4.472364,0.538344,0.315951,0.266104,0.178681,0.290644,0.241564,0.305982


# Data Pre-processing

## Separating Features and Target

In [10]:
# Remove the target and columns that should not be used as predictive features.
# PatientID is an identifier and DoctorInCharge is a non-predictive text column.
X = parkinsons_data.drop(columns=['PatientID', 'Diagnosis', 'DoctorInCharge'])
Y = parkinsons_data['Diagnosis']

print("Feature count:", X.shape[1])
print("Target classes:", sorted(Y.unique()))


Feature count: 32
Target classes: [np.int64(0), np.int64(1)]


In [11]:
# Display the input features
X.head()

,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,...,UPDRS,MoCA,FunctionalAssessment,Tremor,Rigidity,Bradykinesia,PosturalInstability,SpeechProblems,SleepDisorders,Constipation
0,85,0,3,1,19.619878,0,5.108241,1.380660,3.893969,9.283194,...,6.458713,29.181289,1.572427,1,0,0,0,0,0,0
1,75,0,0,2,16.247339,1,6.027648,8.409804,8.513428,5.602470,...,37.306703,12.332639,4.787551,0,1,0,1,0,1,0
2,70,1,0,0,15.368239,0,2.242135,0.213275,6.498805,9.929824,...,67.838170,29.927783,2.130686,1,0,0,0,1,0,1
3,52,0,0,0,15.454557,0,5.997788,1.375045,6.715033,4.196189,...,52.964696,21.304268,3.391288,1,1,1,0,0,0,1
4,87,0,0,1,18.616042,0,9.775243,1.188607,4.657572,9.363925,...,21.804880,8.336364,3.200969,0,0,0,1,0,1,0


In [12]:
# Display the target values
Y.head()

,Diagnosis
0,0
1,1
2,1
3,1
4,0


## Splitting the Data into Training and Test Sets

In [13]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2, stratify=Y
)


In [14]:
print("Full dataset:", X.shape)
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)


Full dataset: (2105, 32)
Training features: (1684, 32)
Testing features: (421, 32)


## Data Standardization

In [15]:
import os, joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn import svm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", svm.SVC(kernel="linear", class_weight="balanced",
                      probability=True, random_state=2)),
])
pipe.fit(X_train, Y_train)

print("Train accuracy:", accuracy_score(Y_train, pipe.predict(X_train)))
pred = pipe.predict(X_test)
print("Test accuracy:", accuracy_score(Y_test, pred))
print(classification_report(Y_test, pred, zero_division=0))
print(confusion_matrix(Y_test, pred))

os.makedirs("models", exist_ok=True)
joblib.dump({"pipeline": pipe, "features": list(X.columns),
             "defaults": X.median().to_dict()}, "models/parkinsons.joblib")
print("Saved models/parkinsons.joblib")

Train accuracy: 0.8248218527315915
Test accuracy: 0.8076009501187649
              precision    recall  f1-score   support

           0       0.72      0.82      0.76       160
           1       0.88      0.80      0.84       261

    accuracy                           0.81       421
   macro avg       0.80      0.81      0.80       421
weighted avg       0.82      0.81      0.81       421

[[131  29]
 [ 52 209]]
Saved models/parkinsons.joblib


In [16]:
# Create the standardization object
scaler = StandardScaler()

In [17]:
# Fit the scaler only on the training data
scaler.fit(X_train)

StandardScaler()

In [18]:
# Apply the same scaling to training and test data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [19]:
# Preview standardized training data
print(X_train_scaled[:5])

[[-0.40654321  0.98936768 -0.68773158  1.85861532 -0.1262226  -0.65909959
  -0.58000781 -1.62715437  0.09282378 -0.38364194 -0.41557424 -0.34809736
  -0.41655632 -0.42047526 -0.50852229 -0.22332781 -1.28075663 -1.12623034
  -0.99972242  0.78947002 -0.0131646   1.25474236 -0.03187691  0.68166438
  -0.14215399 -0.87264327 -0.59838492 -0.5066719  -0.38866369  1.56325391
  -0.5746075   1.54763086]
 [-0.23540991 -1.01074658 -0.68773158 -0.37783212  0.91419956  1.5172214
  -1.57338431 -1.69585714 -0.04276455 -0.36275697  2.40630891 -0.34809736
  -0.41655632 -0.42047526 -0.50852229 -0.22332781 -1.16663048  0.34516667
  -1.77689651 -0.33138514 -0.51911754  0.15514386  1.24309802 -0.23103086
  -1.14858516 -0.87264327 -0.59838492 -0.5066719   2.57291848 -0.63969135
   1.7403184   1.54763086]
 [ 0.19242334  0.98936768 -0.68773158 -0.37783212  0.32921371 -0.65909959
   0.03210183 -0.56981987 -0.95340298 -1.58116206 -0.41557424  2.87275952
  -0.41655632 -0.42047526  1.96648214 -0.22332781  0.811556

# Model Training

## Support Vector Machine (SVM) Model

In [20]:
model = svm.SVC(kernel='linear', class_weight='balanced', random_state=2)

In [21]:
# Train the SVM model
model.fit(X_train_scaled, Y_train)

SVC(class_weight='balanced', kernel='linear', random_state=2)

# Model Evaluation

## Accuracy and Classification Metrics

In [22]:
# Accuracy on training data
X_train_prediction = model.predict(X_train_scaled)
training_data_accuracy = accuracy_score(Y_train, X_train_prediction)


In [23]:
print('Accuracy score on training data:', training_data_accuracy)

Accuracy score on training data: 0.8248218527315915


In [24]:
# Accuracy on test data
X_test_prediction = model.predict(X_test_scaled)
test_data_accuracy = accuracy_score(Y_test, X_test_prediction)


In [25]:
print('Accuracy score on test data:', test_data_accuracy)

print("\nClassification report on test data:")
print(classification_report(Y_test, X_test_prediction, zero_division=0))

print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(Y_test, X_test_prediction))


Accuracy score on test data: 0.8076009501187649

Classification report on test data:
              precision    recall  f1-score   support

           0       0.72      0.82      0.76       160
           1       0.88      0.80      0.84       261

    accuracy                           0.81       421
   macro avg       0.80      0.81      0.80       421
weighted avg       0.82      0.81      0.81       421

Confusion matrix (rows = actual, columns = predicted):
[[131  29]
 [ 52 209]]


# Building a Predictive System

The following example predicts the diagnosis for one record from the dataset. Replace `sample_input` with a dictionary of values for the same feature columns to test another case.

**Note:** This is an educational machine-learning demonstration, not a medical diagnosis. Consult a qualified healthcare professional for medical concerns.

In [26]:
sample_input = X.iloc[[0]].copy()
prediction = pipe.predict(sample_input)
print("Predicted Diagnosis:", int(prediction[0]))

Predicted Diagnosis: 0


In [27]:
from google.colab import files
files.download("models/parkinsons.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>